1. Загрузите данные об описаниях вакансий и соответствующих годовых зарплатах из файла salary-train.csv.
2. Проведите предобработку:
• Приведите тексты к нижнему регистру.
• Замените все, кроме букв и цифр, на пробелы — это облегчит
дальнейшее разделение текста на слова. 
• Замените пропуски в столбцах LocationNormalized и ContractTime
на специальную строку ’nan’. Код для этого был приведен выше


In [ ]:
import pandas as pd
import re
from sklearn.linear_model import Ridge
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

data_train = pd.read_csv('./salary-train.csv')
data_train['LocationNormalized'].fillna('nan', inplace=True)
data_train['ContractTime'].fillna('nan', inplace=True)

data_train = data_train.map(lambda x: re.sub('[^a-zA-Z0-9]', ' ', str(x).lower())) 

• Примените TfidfVectorizer для преобразования текстов в векторы признаков. Оставьте только те слова, которые встречаются хотя бы в 5 объектах

In [ ]:
tfidf = TfidfVectorizer(min_df=5)
X_tfidf = tfidf.fit_transform(data_train['FullDescription'])

• Примените DictVectorizer для получения one-hot-кодирования
признаков LocationNormalized и ContractTime.

In [ ]:
enc = DictVectorizer()
X_train_categ = enc.fit_transform(
    data_train[['LocationNormalized', 'ContractTime']].to_dict('records')
)

• Объедините все полученные признаки в одну матрицу "объекты-признаки".

In [ ]:
X_train = hstack([X_tfidf, X_train_categ])

3. Обучите гребневую регрессию с параметром alpha=1. 

In [ ]:
reg = Ridge(alpha=1)
reg.fit(X_train, data_train['SalaryNormalized'])

4. Постройте прогнозы для двух примеров из файла salary-test-mini.csv.
Значения полученных прогнозов являются ответом на задание

In [ ]:
data_test = pd.read_csv('./salary-test-mini.csv')
data_test['FullDescription'] = data_test['FullDescription'].map(lambda x: re.sub('[^a-zA-Z0-9]', ' ', str(x).lower()))
X_tfidf_test = tfidf.transform(data_test['FullDescription'])
data_test['LocationNormalized'].fillna('nan', inplace=True)
data_test['ContractTime'].fillna('nan', inplace=True)

X_categ_test = enc.transform(data_test[['LocationNormalized', 'ContractTime']].to_dict('records'))
X_test = hstack([X_tfidf_test, X_categ_test])
print(*reg.predict(X_test))